In [ ]:
import numpy as np

# Parámetros del sistema
b = 1/5
a = 0
c = 1/2

# Coeficientes de la ecuación cúbica x^3 + px + q = 0
p = -3 * (1 - 1/b)
q = -3 * a / b

# Calculamos las raíces del polinomio
raices = np.roots([1, 0, p, q])
print(f"Raices obtenidas: {np.round(raices, 4)}\n")

def clasificar_equilibrio(vaps):
    re = np.real(vaps)

    if np.any(re > 0) and np.any(re < 0):
        return "Silla"
    elif np.all(re < 0):
        return "Atractor"
    elif np.all(re > 0):
        return "Repulsor"
    elif np.all(np.isclose(re, 0)):
        return "Neutro"

    return "Indeterminado"

equilibrios_reales = []

for x in raices:
    # Nos quedamos solo con las raíces reales (ignorando el ruído numérico imaginario)
    if not np.isclose(np.imag(x), 0, atol=1e-10):
        continue

    x_real = np.real(x)

    # Matriz Jacobiana evaluada en la raíz
    A = np.array([
        [c * (1 - x_real**2), c],
        [-1/c, -b/c]
    ])

    vaps = np.linalg.eigvals(A)
    tipo = clasificar_equilibrio(vaps)

    equilibrios_reales.append((x_real, A, vaps, tipo))

# Salida de resultados
print("Resumen:")

if not equilibrios_reales:
    print("No se han encontrado puntos de equilibrio reales.")
else:
    for x_real, A, vaps, tipo in equilibrios_reales:
        print(f"x = {x_real}")
        print(f"Tipo: {tipo}")
        print(f"Valores propios: {vaps}")
        print("-" * 40)

Raices obtenidas: [-0.+3.4641j  0.-3.4641j  0.+0.j    ]

Resumen:
x = 0.0
Tipo: Repulsor
Valores propios: [0.05+0.89302855j 0.05-0.89302855j]
----------------------------------------


cte de lyapunov primera parte


In [ ]:
import sympy as sp

# Variables y parámetros
u, v = sp.symbols('u v', real=True)
b, c, x = sp.symbols('b c x', real=True)

# Ecuaciones del modelo original
f = c * (u + v - (u**3)/3 - (u**2)*x - u*(x**2))
g = -(u + b*v) / c

# Función auxiliar para derivar y evaluar en el origen
def derivar_en_origen(funcion, variables):
    expr = funcion
    for var in variables:
        expr = sp.diff(expr, var)
    return expr.subs({u: 0, v: 0})

# Derivadas de segundo orden
f_uu = derivar_en_origen(f, [u, u])
f_uv = derivar_en_origen(f, [u, v])
f_vv = derivar_en_origen(f, [v, v])

g_uu = derivar_en_origen(g, [u, u])
g_uv = derivar_en_origen(g, [u, v])
g_vv = derivar_en_origen(g, [v, v])

# Derivadas de tercer orden
f_uuu = derivar_en_origen(f, [u, u, u])
f_uvv = derivar_en_origen(f, [u, v, v])
g_uuv = derivar_en_origen(g, [u, u, v])
g_vvv = derivar_en_origen(g, [v, v, v])

# Jacobiano para calcular la frecuencia (omega) a partir de los valores propios
J = sp.Matrix([
    [sp.diff(f, u), sp.diff(f, v)],
    [sp.diff(g, u), sp.diff(g, v)]
]).subs({u: 0, v: 0})

valores_propios = list(J.eigenvals().keys())
omega = sp.sqrt(-valores_propios[0])

# Calculo de l1
termino1 = (f_uuu + f_uvv + g_uuv + g_vvv) / 16
termino2 = (f_uv*(f_uu + f_vv) - g_uv*(g_uu + g_vv) - f_uu*g_uu + f_vv*g_vv) / (16 * omega)

l1 = sp.simplify(termino1 + termino2)
print("Primer coeficiente de Lyapunov l1:", l1)


Primer coeficiente de Lyapunov l1: -c/8


constante de lyapunov segunda parte b=0 c=+-1

In [ ]:
import sympy as sp

# Variables y parámetros
u, v = sp.symbols('u v', real=True)
eps = sp.symbols('eps', positive=True)
c_param = sp.symbols('c_param', real=True)

# Punto de equilibrio trasladado
x_star = c_param
y_star = c_param - (c_param**3)/3

# Sistema modificado
f = (u + x_star) - (v + y_star) - ((u + x_star)**3)/3
g = eps * ((u + x_star) - c_param)

# Función auxiliar
def derivar_en_origen(funcion, variables):
    expr = funcion
    for var in variables:
        expr = sp.diff(expr, var)
    return expr.subs({u: 0, v: 0})

# Calculamos las derivadas necesarias en (0,0)
f_uu = derivar_en_origen(f, [u, u])
f_uv = derivar_en_origen(f, [u, v])
f_vv = derivar_en_origen(f, [v, v])
g_uu = derivar_en_origen(g, [u, u])
g_uv = derivar_en_origen(g, [u, v])
g_vv = derivar_en_origen(g, [v, v])

f_uuu = derivar_en_origen(f, [u, u, u])
f_uvv = derivar_en_origen(f, [u, v, v])
g_uuv = derivar_en_origen(g, [u, u, v])
g_vvv = derivar_en_origen(g, [v, v, v])

# Omega para la formula general
omega = sp.symbols('omega', positive=True)

termino1 = (f_uuu + f_uvv + g_uuv + g_vvv) / 16
termino2 = (f_uv*(f_uu + f_vv) - g_uv*(g_uu + g_vv) - f_uu*g_uu + f_vv*g_vv) / (16 * omega)

l1 = sp.simplify(termino1 + termino2)
print("Primer coeficiente de Lyapunov l1 general:", l1)

Primer coeficiente de Lyapunov l1 general: -1/8


In [ ]:

import sympy as sp

# Caso particular con c=0 y b distinto de 0
u, v = sp.symbols('u v', real=True)
eps = sp.symbols('eps', positive=True)
b_param = sp.symbols('b_param', real=True)
x_star = sp.symbols('x_star', real=True)

# Relación en el equilibrio
y_star = x_star / b_param

# Sistema
f = (u + x_star) - (v + y_star) - ((u + x_star)**3)/3
g = eps * ((u + x_star) - b_param*(v + y_star))

def derivar_en_origen(funcion, variables):
    expr = funcion
    for var in variables:
        expr = sp.diff(expr, var)
    return expr.subs({u: 0, v: 0})

f_uu = derivar_en_origen(f, [u, u])
f_uv = derivar_en_origen(f, [u, v])
f_vv = derivar_en_origen(f, [v, v])
g_uu = derivar_en_origen(g, [u, u])
g_uv = derivar_en_origen(g, [u, v])
g_vv = derivar_en_origen(g, [v, v])

f_uuu = derivar_en_origen(f, [u, u, u])
f_uvv = derivar_en_origen(f, [u, v, v])
g_uuv = derivar_en_origen(g, [u, u, v])
g_vvv = derivar_en_origen(g, [v, v, v])

omega = sp.symbols('omega', positive=True)

termino1 = (f_uuu + f_uvv + g_uuv + g_vvv) / 16
termino2 = (f_uv*(f_uu + f_vv) - g_uv*(g_uu + g_vv) - f_uu*g_uu + f_vv*g_vv) / (16 * omega)

l1 = sp.simplify(termino1 + termino2)
print("Primer coeficiente de Lyapunov l1 (caso b!=0):", l1)

Primer coeficiente de Lyapunov l1 (caso b!=0): -1/8
